# Phase 1: Input Validation & Molecular Preprocessing

Goals:
- load the Source and Target datasets
- standardise SMILES
- remove duplicates
- generate basic statistics
- compute source–target overlap

In [1]:
import os
import sys
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import inchi
import json
from pathlib import Path

# Path settings
ROOT = r'..'
DATA_DIR = os.path.join(ROOT, 'data', 'cv_datasets')
BENCHMARK_DIR = os.path.join(ROOT, 'benchmark')
RESULTS_DIR = os.path.join(BENCHMARK_DIR, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

print("="*100)
print("PHASE 1: Input Validation & Molecular Preprocessing")
print("="*100)
print(f"Root: {ROOT}")
print(f"Data: {DATA_DIR}")
print(f"Results: {RESULTS_DIR}\n")

PHASE 1: Input Validation & Molecular Preprocessing
Root: ..
Data: ..\data\cv_datasets
Results: ..\benchmark\results



In [2]:
def convert_to_python_types(obj):
    """Convert numpy types to native Python types for JSON serialization"""
    if isinstance(obj, dict):
        return {k: convert_to_python_types(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_python_types(item) for item in obj]
    elif isinstance(obj, (np.integer, np.floating)):
        return float(obj) if isinstance(obj, np.floating) else int(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

def get_smiles_col(df):
    if 'smiles' in df.columns:
        return 'smiles'
    elif 'Drug' in df.columns:
        return 'Drug'
    else:
        return None

def get_value_col(df):
    if 'value' in df.columns:
        return 'value'
    elif 'Y' in df.columns:
        return 'Y'
    else:
        return None

def standardize_smiles(smiles_str):
    """Convert SMILES to canonical form"""
    try:
        mol = Chem.MolFromSmiles(str(smiles_str))
        if mol is not None:
            return Chem.MolToSmiles(mol)
        else:
            return None
    except:
        return None

def get_inchi_key(smiles_str):
    """Generate an InChIKey from SMILES"""
    try:
        mol = Chem.MolFromSmiles(str(smiles_str))
        if mol is not None:
            return inchi.MolToInchiKey(mol)
        else:
            return None
    except:
        return None

print("✓ Helper functions defined")

✓ Helper functions defined


In [3]:
# Load the datasets
PROPERTIES = ['clearance', 'half_life', 'fu']
FOLD = 0  # use only the first fold

source_data = {}
target_data = {}

fold_dir = os.path.join(DATA_DIR, f'fold_{FOLD}')

for prop in PROPERTIES:
    print(f"Loading {prop}...")
    
    # Source
    src_path = os.path.join(fold_dir, f'{prop}_source.csv')
    df_src = pd.read_csv(src_path)
    source_data[prop] = df_src.copy()
    print(f"  Source: {len(df_src)} rows")
    
    # Target train
    tgt_path = os.path.join(fold_dir, f'{prop}_target_train.csv')
    df_tgt = pd.concat([pd.read_csv(os.path.join(fold_dir, f'{prop}_target_{s}.csv')) for s in ('train', 'test')],
                       ignore_index=True)  # the gap analysis uses the whole target (train+test), not a single fold
    target_data[prop] = df_tgt.copy()
    print(f"  Target: {len(df_tgt)} rows")

print("\n✓ Datasets loaded")

Loading clearance...
  Source: 14387 rows
  Target: 1027 rows
Loading half_life...
  Source: 9204 rows
  Target: 1164 rows
Loading fu...
  Source: 3783 rows
  Target: 353 rows

✓ Datasets loaded


In [4]:
# Molecular Preprocessing
preprocessing_results = {}

for prop in PROPERTIES:
    print(f"\n{'='*80}")
    print(f"Property: {prop.upper()}")
    print(f"{'='*80}")
    
    preprocessing_results[prop] = {}
    
    for domain_type, data_dict in [('source', source_data), ('target', target_data)]:
        df = data_dict[prop].copy()
        print(f"\n  {domain_type.upper()}:")
        
        original_rows = len(df)
        print(f"    Original: {original_rows} rows")
        
        # SMILES column detection
        smiles_col = get_smiles_col(df)
        value_col = get_value_col(df)
        print(f"    SMILES column: {smiles_col}")
        print(f"    Value column: {value_col}")
        
        # Remove missing SMILES
        df = df.dropna(subset=[smiles_col])
        missing_smiles = original_rows - len(df)
        print(f"    Missing SMILES: {missing_smiles}")
        
        # Standardize SMILES
        print(f"    Standardizing SMILES...")
        df['smiles_canonical'] = df[smiles_col].apply(standardize_smiles)
        
        # Remove invalid SMILES
        invalid = df['smiles_canonical'].isna().sum()
        df = df[df['smiles_canonical'].notna()].copy()
        print(f"    Invalid SMILES: {invalid}")
        
        # Generate InChIKey
        print(f"    Generating InChIKey...")
        df['inchi_key'] = df['smiles_canonical'].apply(get_inchi_key)
        
        # Check duplicates
        unique_inchi = df['inchi_key'].nunique()
        duplicate_by_inchi = len(df) - unique_inchi
        print(f"    Total valid: {len(df)}")
        print(f"    Unique by InChIKey: {unique_inchi}")
        print(f"    Duplicates: {duplicate_by_inchi}")
        
        # Store preprocessed data
        data_dict[prop] = df[[smiles_col, value_col, 'smiles_canonical', 'inchi_key']].copy()
        data_dict[prop].columns = ['smiles_original', 'value', 'smiles', 'inchi_key']
        
        preprocessing_results[prop][domain_type] = {
            'original_rows': int(original_rows),
            'missing_smiles': int(missing_smiles),
            'invalid_smiles': int(invalid),
            'final_rows': int(len(df)),
            'unique_inchi': int(unique_inchi),
            'duplicates': int(duplicate_by_inchi),
        }

print("\n✓ Molecular preprocessing complete")


Property: CLEARANCE

  SOURCE:
    Original: 14387 rows
    SMILES column: smiles
    Value column: value
    Missing SMILES: 0
    Standardizing SMILES...
    Invalid SMILES: 0
    Generating InChIKey...
    Total valid: 14387
    Unique by InChIKey: 13564
    Duplicates: 823

  TARGET:
    Original: 1027 rows
    SMILES column: smiles
    Value column: value
    Missing SMILES: 0
    Standardizing SMILES...
    Invalid SMILES: 0
    Generating InChIKey...
    Total valid: 1027
    Unique by InChIKey: 1027
    Duplicates: 0

Property: HALF_LIFE

  SOURCE:
    Original: 9204 rows
    SMILES column: smiles
    Value column: value
    Missing SMILES: 0
    Standardizing SMILES...
    Invalid SMILES: 0
    Generating InChIKey...
    Total valid: 9204
    Unique by InChIKey: 8477
    Duplicates: 727

  TARGET:
    Original: 1164 rows
    SMILES column: smiles
    Value column: value
    Missing SMILES: 0
    Standardizing SMILES...
    Invalid SMILES: 0
    Generating InChIKey...
    Tota

In [5]:
# Compute source–target overlap
print(f"\n{'='*80}")
print("SOURCE-TARGET OVERLAP ANALYSIS")
print(f"{'='*80}\n")

overlap_results = {}

for prop in PROPERTIES:
    print(f"{prop.upper()}:")
    
    src_inchi = set(source_data[prop]['inchi_key'].unique())
    tgt_inchi = set(target_data[prop]['inchi_key'].unique())
    
    overlap_inchi = src_inchi & tgt_inchi
    src_only = src_inchi - tgt_inchi
    tgt_only = tgt_inchi - src_inchi
    
    overlap_pct_src = 100.0 * len(overlap_inchi) / len(src_inchi) if len(src_inchi) > 0 else 0
    overlap_pct_tgt = 100.0 * len(overlap_inchi) / len(tgt_inchi) if len(tgt_inchi) > 0 else 0
    
    print(f"  Source unique molecules: {len(src_inchi)}")
    print(f"  Target unique molecules: {len(tgt_inchi)}")
    print(f"  Overlap: {len(overlap_inchi)} ({overlap_pct_src:.1f}% of source, {overlap_pct_tgt:.1f}% of target)")
    print(f"  Source-only: {len(src_only)}")
    print(f"  Target-only: {len(tgt_only)}")
    
    # Compare labels of overlapping compounds
    if len(overlap_inchi) > 0:
        overlap_src = source_data[prop][source_data[prop]['inchi_key'].isin(overlap_inchi)]
        overlap_tgt = target_data[prop][target_data[prop]['inchi_key'].isin(overlap_inchi)]
        
        # Merge on InChIKey
        merged = pd.merge(overlap_src[['inchi_key', 'value']], 
                         overlap_tgt[['inchi_key', 'value']], 
                         on='inchi_key', 
                         suffixes=('_src', '_tgt'))
        
        if len(merged) > 0:
            r_pearson = float(merged[['value_src', 'value_tgt']].corr().iloc[0, 1])
            r_spearman = float(merged[['value_src', 'value_tgt']].corr(method='spearman').iloc[0, 1])
            r2 = float(r_pearson ** 2)
            rmse = float(np.sqrt(np.mean((merged['value_src'] - merged['value_tgt'])**2)))
            mbe = float(np.mean(merged['value_src'] - merged['value_tgt']))
            
            print(f"  Overlap label correlation (n={len(merged)}):")
            print(f"    Pearson r: {r_pearson:.4f}")
            print(f"    Spearman ρ: {r_spearman:.4f}")
            print(f"    R²: {r2:.4f}")
            print(f"    RMSE: {rmse:.4f}")
            print(f"    MBE: {mbe:.4f}")
            
            overlap_results[prop] = {
                'overlap_count': int(len(overlap_inchi)),
                'overlap_pct_src': float(overlap_pct_src),
                'overlap_pct_tgt': float(overlap_pct_tgt),
                'label_correlation_count': int(len(merged)),
                'pearson_r': r_pearson,
                'spearman_rho': r_spearman,
                'r2': r2,
                'rmse': rmse,
                'mbe': mbe,
            }
    else:
        print(f"  No overlap compounds found")
        overlap_results[prop] = {
            'overlap_count': 0,
            'label_correlation_count': 0,
        }
    
    print()


SOURCE-TARGET OVERLAP ANALYSIS

CLEARANCE:
  Source unique molecules: 13564
  Target unique molecules: 1027
  Overlap: 0 (0.0% of source, 0.0% of target)
  Source-only: 13564
  Target-only: 1027
  No overlap compounds found

HALF_LIFE:
  Source unique molecules: 8477
  Target unique molecules: 1164
  Overlap: 0 (0.0% of source, 0.0% of target)
  Source-only: 8477
  Target-only: 1164
  No overlap compounds found

FU:
  Source unique molecules: 3597
  Target unique molecules: 353
  Overlap: 0 (0.0% of source, 0.0% of target)
  Source-only: 3597
  Target-only: 353
  No overlap compounds found



In [6]:
# Summary Report
print(f"\n{'='*100}")
print("SUMMARY REPORT")
print(f"{'='*100}\n")

summary_data = []

for prop in PROPERTIES:
    row = {
        'Property': prop,
        'Source (orig)': preprocessing_results[prop]['source']['original_rows'],
        'Source (final)': preprocessing_results[prop]['source']['final_rows'],
        'Target (orig)': preprocessing_results[prop]['target']['original_rows'],
        'Target (final)': preprocessing_results[prop]['target']['final_rows'],
        'Overlap': overlap_results[prop].get('overlap_count', 0),
        'Overlap %': f"{overlap_results[prop].get('overlap_pct_src', 0):.1f}%",
    }
    summary_data.append(row)

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))
print()

# Save results
with open(os.path.join(RESULTS_DIR, 'phase_01_preprocessing.json'), 'w') as f:
    json.dump(convert_to_python_types(preprocessing_results), f, indent=2)

with open(os.path.join(RESULTS_DIR, 'phase_01_overlap.json'), 'w') as f:
    json.dump(convert_to_python_types(overlap_results), f, indent=2)

summary_df.to_csv(os.path.join(RESULTS_DIR, 'phase_01_summary.csv'), index=False)

print("✓ Results saved to benchmark/results/")
print(f"  - phase_01_preprocessing.json")
print(f"  - phase_01_overlap.json")
print(f"  - phase_01_summary.csv")


SUMMARY REPORT

 Property  Source (orig)  Source (final)  Target (orig)  Target (final)  Overlap Overlap %
clearance          14387           14387           1027            1027        0      0.0%
half_life           9204            9204           1164            1164        0      0.0%
       fu           3783            3783            353             353        0      0.0%

✓ Results saved to benchmark/results/
  - phase_01_preprocessing.json
  - phase_01_overlap.json
  - phase_01_summary.csv


In [7]:
# Save preprocessed data for next phases
import pickle

with open(os.path.join(RESULTS_DIR, 'preprocessed_data.pkl'), 'wb') as f:
    pickle.dump({
        'source_data': source_data,
        'target_data': target_data,
        'properties': PROPERTIES,
        'fold': FOLD,
    }, f)

print("\n✓ Preprocessed data saved for Phase 2")
print("\nPhase 1 Complete! Ready for Phase 2: Chemical Space Analysis")


✓ Preprocessed data saved for Phase 2

Phase 1 Complete! Ready for Phase 2: Chemical Space Analysis
